In [4]:
import pandas as pd  # 표 형태 데이터(DataFrame)를 다루는 라이브러리
import requests      # 웹 서버에 요청을 보내는 라이브러리
from bs4 import BeautifulSoup  # HTML 문서에서 필요한 요소를 찾는 라이브러리

print("pandas version:", pd.__version__)

pandas version: 3.0.3


---
## 1. requests로 요청과 응답 확인하기

- `requests.get()` : URL로 요청 보내기
- `Response` 객체 : 서버 응답 결과
- `status_code` : 응답 상태 코드
- `200` : 정상 응답
- `404` : 요청한 페이지를 찾을 수 없음
- `response.text` : 응답 본문 확인
- `timeout=5` : 서버 응답을 최대 5초만 기다림


In [5]:
# URL 요청하기
# timeout=5: 서버 응답을 최대 5초만 기다림
#              응답이 없을 때 코드가 계속 멈춰 있지 않도록 제한

response = requests.get("https://google.com", timeout=5)

print("응답 객체:", type(response))
print("상태 코드:", response.status_code)
print("응답 이유:", response.reason)
print("응답 형식:", response.headers.get("Content-Type"))

응답 객체: <class 'requests.models.Response'>
상태 코드: 200
응답 이유: OK
응답 형식: text/html; charset=ISO-8859-1


In [6]:
# 응답 내용 일부 확인
# response.text: 서버가 보내준 HTML/텍스트 응답 본문
print(response.text[:500])

<!doctype html><html itemscope="" itemtype="http://schema.org/WebPage" lang="ko"><head><meta content="text/html; charset=UTF-8" http-equiv="Content-Type"><meta content="/images/branding/googleg/1x/googleg_standard_color_128dp.png" itemprop="image"><title>Google</title><script nonce="ZCNREBsqYHeJU4ZlWoH2GA">(function(){var _g={kEI:'PXUzaoaRMvCf0PEPypSz0AI',kEXPI:'0,1304203,2935842,14111,34680,30022,360901,5520325,11,12259,12,193,36811568,25228681,152385,65167,138347,37798,77237,38125,33385,26230,


### 요청이 성공하지 않았을 때

- 주소가 잘못되었거나 페이지가 없으면 404가 나올 수 있음
- `status_code`로 서버 응답 상태를 확인할 수 있음

In [7]:
# 잘못된 주소 요청하기
# timeout=5: 서버 응답을 최대 5초만 기다림
bad_url = "https://google.com/wrong-page-for-practice"  # 크롤링 url 상태 확인할 때 
response = requests.get(bad_url, timeout=5)

print("상태 코드:", response.status_code)
print("응답 이유:", response.reason)


상태 코드: 404
응답 이유: Not Found


In [8]:
# 수동으로 상태 코드 확인하기
if response.status_code == 200:
    print("status ok")
else:
    print("status error", response.status_code)

status error 404


In [9]:
# 자동으로 오류 상태 확인하기
# raise_for_status(): 상태 코드가 4xx/5xx이면 예외 발생
# 이 셀은 404 예제라 실행하면 오류가 발생하는 것이 정상
response = requests.get(bad_url, timeout=5)
response.raise_for_status()


HTTPError: 404 Client Error: Not Found for url: https://google.com/wrong-page-for-practice

In [ ]:
# 실무에서는 크롤링이 중간에 멈추지 않도록 try/except로 처리
try:
    response = requests.get(bad_url, timeout=5)
    response.raise_for_status()
    print("요청 성공")
except requests.exceptions.HTTPError as e:
    print("HTTP 오류 발생:", e)


### User-Agent 지정하기

- `User-Agent`는 서버에 전달되는 **클라이언트 정보**
- 같은 URL이라도 PC 브라우저, 모바일 브라우저, 자동화 프로그램에 따라 서버 응답이 달라질 수 있음
- 일부 사이트는 크롤러·자동화 요청을 제한하기 위해 `User-Agent`를 확인함
- 필요할 때 `headers`에 `User-Agent`를 넣어 브라우저 요청처럼 전달할 수 있음
- 내 브라우저의 User-Agent는 `what is my user agent`로 검색해서 확인 가능


In [10]:
# User-Agent를 지정해 요청하기
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
headers = {"User-Agent": user_agent}

# headers: 서버에 함께 전달할 요청 정보
# timeout=5: 서버 응답을 최대 5초만 기다림
response = requests.get("https://google.com", headers=headers, timeout=5)

print("상태 코드:", response.status_code)
print("응답 길이:", len(response.text))
print("요청에 사용한 User-Agent:", headers["User-Agent"][:80] + "...")


상태 코드: 200
응답 길이: 223118
요청에 사용한 User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)...


In [11]:
# 응답받은 HTML을 파일로 저장하기
# open(..., "w"): 쓰기 모드로 파일 열기
#with open("mygoogle.html", "w", encoding="utf-8") as f:
#    f.write(response.text)

#print("저장 완료: mygoogle.html")


---
## 2. JSON API 응답을 DataFrame으로 변환하기

- 이번 예제는 `JSONPlaceholder`의 `/users` 데이터를 사용
- `JSONPlaceholder`는 API 연습용 가짜 REST API 서비스
- 회원가입이나 인증키 없이 JSON 응답을 받을 수 있음
- API 응답은 JSON 형식으로 제공되는 경우가 많음
- `response.json()`으로 JSON 응답을 파이썬 객체로 변환
- `pd.DataFrame()`으로 표 형태로 변환

### 응답 형식 확인하기

- `Content-Type`: 서버가 응답 본문이 어떤 형식인지 알려주는 값
- 앞에서 요청한 Google 페이지는 HTML 문서라 보통 `text/html` 형식
- JSON API는 보통 `application/json` 형식


In [12]:
api_url = "https://jsonplaceholder.typicode.com/users"

# timeout=5: 서버 응답을 최대 5초까지 기다림
response = requests.get(api_url, timeout=5)

print("상태 코드:", response.status_code)
print("응답 형식:", response.headers.get("Content-Type"))  # JSON 응답인지 확인


상태 코드: 200
응답 형식: application/json; charset=utf-8


In [13]:
# response.json(): JSON 응답을 파이썬 리스트/딕셔너리로 변환
users = response.json()

print(type(users))
print("데이터 개수:", len(users))
users[0]  # 첫 번째 데이터 확인

<class 'list'>
데이터 개수: 10


{'id': 1,
 'name': 'Leanne Graham',
 'username': 'Bret',
 'email': 'Sincere@april.biz',
 'address': {'street': 'Kulas Light',
  'suite': 'Apt. 556',
  'city': 'Gwenborough',
  'zipcode': '92998-3874',
  'geo': {'lat': '-37.3159', 'lng': '81.1496'}},
 'phone': '1-770-736-8031 x56442',
 'website': 'hildegard.org',
 'company': {'name': 'Romaguera-Crona',
  'catchPhrase': 'Multi-layered client-server neural-net',
  'bs': 'harness real-time e-markets'}}

In [14]:
# 필요한 항목만 골라 표 형태로 정리
user_rows = []

for user in users:
    user_rows.append({
        "id": user.get("id"),
        "name": user.get("name"),
        "username": user.get("username"),
        "email": user.get("email"),
        "city": user.get("address", {}).get("city"),
        "company": user.get("company", {}).get("name")
    })



# DataFrame: 행과 열이 있는 표 형태 데이터
df_users = pd.DataFrame(user_rows)

df_users.head()  # 앞부분 몇 줄 확인

,id,name,username,email,city,company
0,1,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,South Elvis,Robel-Corkery
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC


### API 요청 조건을 `params`로 전달하기

- 실제 API는 URL만 보내는 것이 아니라 검색 조건을 함께 보내는 경우가 많음
- `params`는 요청 조건을 딕셔너리로 전달하는 방식
- 코드가 깔끔하고, 조건을 바꿔가며 요청하기 쉬움
- 공공데이터 API에서는 인증키, 페이지 번호, 검색어 등을 `params`에 넣는 경우가 많음

In [15]:
# id가 1인 사용자만 요청하기
params = {"id": 1}

# params: API에 전달할 요청 조건
# timeout=5: 서버 응답을 최대 5초만 기다림
response = requests.get(api_url, params=params, timeout=5)

print("요청 URL:", response.url)
print("상태 코드:", response.status_code)

user_one = response.json()
user_one

요청 URL: https://jsonplaceholder.typicode.com/users?id=1
상태 코드: 200


[{'id': 1,
  'name': 'Leanne Graham',
  'username': 'Bret',
  'email': 'Sincere@april.biz',
  'address': {'street': 'Kulas Light',
   'suite': 'Apt. 556',
   'city': 'Gwenborough',
   'zipcode': '92998-3874',
   'geo': {'lat': '-37.3159', 'lng': '81.1496'}},
  'phone': '1-770-736-8031 x56442',
  'website': 'hildegard.org',
  'company': {'name': 'Romaguera-Crona',
   'catchPhrase': 'Multi-layered client-server neural-net',
   'bs': 'harness real-time e-markets'}}]

In [16]:
# params 결과도 DataFrame으로 변환 가능
# DataFrame: 표 형태 데이터
df_user_one = pd.DataFrame(user_one)

df_user_one[["id", "name", "username", "email"]]  # 필요한 열만 선택

,id,name,username,email
0,1,Leanne Graham,Bret,Sincere@april.biz


---
## 3. FinanceDataReader로 주가 데이터 수집하기

- 앞에서는 `requests`로 직접 URL에 요청을 보내고 응답을 확인함
- 여기서는 금융 데이터 수집용 라이브러리인 `FinanceDataReader`를 사용
- 내부 요청 과정을 직접 작성하지 않아도, 종목코드와 기간만으로 주가 데이터를 가져올 수 있음
- 즉, 외부 데이터를 수집하되 더 편리하게 감싼 방식
- 설치가 필요하면 CMD 또는 Anaconda Prompt에서 `pip install finance-datareader` 실행
- 설치 후 노트북 커널을 재시작하고 진행


In [24]:
# 최초 1회 설치가 필요할 수 있음
# CMD 또는 Anaconda Prompt에서 실행: pip install finance-datareader

import FinanceDataReader as fdr

stock_code = "005930"  # 삼성전자
start_date = "2024-01-01"
end_date = "2024-03-31"

# DataReader: 종목코드와 기간을 넣어 주가 데이터를 외부에서 가져옴
df_stock = fdr.DataReader(stock_code, start_date, end_date)

df_stock.head()  # 앞부분 몇 줄 확인

,Open,High,Low,Close,Volume,Change
Date,,,,,,
2024-01-02,78200,79800,78200,79600,17142847,0.014013
2024-01-03,78500,78800,77000,77000,21753644,-0.032663
2024-01-04,76100,77300,76100,76600,15324439,-0.005195
2024-01-05,76700,77100,76400,76600,11304316,0.000000
2024-01-08,77000,77500,76400,76500,11088724,-0.001305


In [27]:
# reset_index(): 날짜 인덱스를 일반 열로 바꿈
df_stock = df_stock.reset_index()

# Date 컬럼 정리
if "Date" not in df_stock.columns:
    df_stock = df_stock.rename(columns={df_stock.columns[0]: "Date"})

# to_datetime(): 문자열/날짜 데이터를 날짜형으로 변환
df_stock["Date"] = pd.to_datetime(df_stock["Date"])

# info(): 컬럼, 결측값, 자료형 확인
df_stock.info()

<class 'pandas.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   index   61 non-null     int64         
 1   Date    61 non-null     datetime64[us]
 2   Open    61 non-null     int64         
 3   High    61 non-null     int64         
 4   Low     61 non-null     int64         
 5   Close   61 non-null     int64         
 6   Volume  61 non-null     int64         
 7   Change  61 non-null     float64       
dtypes: datetime64[us](1), float64(1), int64(6)
memory usage: 3.9 KB


In [28]:
print("기간:", df_stock["Date"].min(), "~", df_stock["Date"].max())

# 필요한 열만 선택해서 앞부분 확인
df_stock[["Date", "Open", "High", "Low", "Close", "Volume"]].head()

기간: 2024-01-02 00:00:00 ~ 2024-03-29 00:00:00


,Date,Open,High,Low,Close,Volume
0,2024-01-02,78200,79800,78200,79600,17142847
1,2024-01-03,78500,78800,77000,77000,21753644
2,2024-01-04,76100,77300,76100,76600,15324439
3,2024-01-05,76700,77100,76400,76600,11304316
4,2024-01-08,77000,77500,76400,76500,11088724


---
## 4. CSV로 저장하기

- 수집한 데이터를 파일로 저장
- `to_csv()` : DataFrame을 CSV 파일로 저장
- `encoding` : 파일을 어떤 문자 규칙으로 저장할지 지정
- `utf-8-sig` : 엑셀에서 한글이 깨지는 것을 줄이기 위해 사용


In [29]:
out_path = "samsung_stock_raw.csv"

# to_csv(): DataFrame을 CSV 파일로 저장
# index=False: 행 번호는 파일에 저장하지 않음
# encoding: 파일을 저장할 때 사용할 문자 규칙
# utf-8-sig: 엑셀에서 CSV를 열 때 한글 깨짐을 줄이기 위해 사용
df_stock.to_csv(out_path, index=False, encoding="utf-8-sig")

print("저장 완료:", out_path)
print("저장 행 수:", len(df_stock))


저장 완료: samsung_stock_raw.csv
저장 행 수: 61


---
## 5. BeautifulSoup 기초 문법

- BeautifulSoup은 HTML 문서에서 필요한 요소를 추출하는 라이브러리
- `find()` : 하나 찾기
- `find_all()` : 여러 개 찾기
- `select()` : CSS 선택자로 찾기
- `get_text()` : 태그 안의 글자 추출
- `태그["속성명"]` : 속성값 추출

### CSS selector 기본

```text
p                 p 태그 선택
.stock            class가 stock인 요소 선택
#main             id가 main인 요소 선택
p.stock           p 태그 중 class가 stock인 요소 선택
div a             div 안쪽의 모든 a 선택
div > a           div 바로 아래의 a 선택
a[href]           href 속성이 있는 a 선택
```


In [32]:
html_doc = """
<html>
  <body>
    <h1 id="main">금융 데이터 사이트 모음</h1>

    <p class="stock">삼성전자</p>
    <p class="stock">카카오</p>
    <p class="stock">네이버</p>

    <a href="https://finance.naver.com" class="finance">네이버 금융</a>
    <a href="https://finance.yahoo.com" class="finance">야후 파이낸스</a>
  </body>
</html>
"""

In [34]:
# BeautifulSoup: HTML 문자열을 파싱 가능한 객체로 변환
soup = BeautifulSoup(html_doc, "html.parser")

In [54]:
type(soup)

print(soup.prettify()) # 보기 편하게 형식을 잡아줌

<html>
 <body>
  <h1 id="main">
   금융 데이터 사이트 모음
  </h1>
  <p class="stock">
   삼성전자
  </p>
  <p class="stock">
   카카오
  </p>
  <p class="stock">
   네이버
  </p>
  <a class="finance" href="https://finance.naver.com">
   네이버 금융
  </a>
  <a class="finance" href="https://finance.yahoo.com">
   야후 파이낸스
  </a>
 </body>
</html>



In [53]:
soup.a

<a class="finance" href="https://finance.naver.com">네이버 금융</a>

In [56]:
[p.get_text() for p in soup.find_all("p")]

['삼성전자', '카카오', '네이버']

In [35]:
print("첫 번째 p 태그:", soup.find("p").get_text())
print("모든 p 태그:", [p.get_text() for p in soup.find_all("p")])
print("class가 stock인 요소:", [x.get_text() for x in soup.select(".stock")])
print("id가 main인 요소:", soup.select_one("#main").get_text())

첫 번째 p 태그: 삼성전자
모든 p 태그: ['삼성전자', '카카오', '네이버']
class가 stock인 요소: ['삼성전자', '카카오', '네이버']
id가 main인 요소: 금융 데이터 사이트 모음


In [62]:
soup.select(".stock") # find_all

[<p class="stock">삼성전자</p>, <p class="stock">카카오</p>, <p class="stock">네이버</p>]

In [64]:
soup.select_one(".stock") # find

<p class="stock">삼성전자</p>

In [77]:
# CSS selector를 활용한 예시
print("p.stock:", [x.get_text() for x in soup.select("p.stock")])
print("body > h1:", soup.select_one("body > h1").get_text())

# a 태그의 텍스트와 href 속성 추출
for a in soup.select("a.finance"):
    print(a.get_text(), "→", a["href"]) # href : 링크 표출


p.stock: ['삼성전자', '카카오', '네이버']
body > h1: 금융 데이터 사이트 모음
네이버 금융 → https://finance.naver.com
야후 파이낸스 → https://finance.yahoo.com


---
## 6. HTML 페이지 요청 후 파싱하기

- `requests.get()`으로 HTML 문서 요청
- `User-Agent`를 넣어 브라우저 요청처럼 전달
- BeautifulSoup으로 HTML 파싱


In [78]:
# 위키백과의 실제 문서 제목을 URL에 사용
# 기존 "나라별_명목_국내_총생산_순위" 주소는 존재하지 않아 404가 발생할 수 있음
wiki_url = "https://ko.wikipedia.org/wiki/명목_국내_총생산순_나라_목록"

# 앞에서 만든 headers(User-Agent)를 그대로 사용
# timeout=5: 서버 응답을 최대 5초만 기다림
response = requests.get(wiki_url, headers=headers, timeout=5)
html = response.text

# HTML 문서를 BeautifulSoup 객체로 변환
soup = BeautifulSoup(html, "html.parser")

print("상태 코드:", response.status_code)
print("페이지 제목:", soup.title.get_text(strip=True) if soup.title else "제목 없음")


상태 코드: 200
페이지 제목: 명목 국내 총생산순 나라 목록 - 위키백과, 우리 모두의 백과사전


---
## 7. HTML 표를 DataFrame으로 만들기

HTML 표는 보통 아래처럼 **제목 행(th)** 과 **데이터 행(td)** 으로 구성됩니다.

```text
<table>
  <tr>  ← 제목 행
    <th>순위</th> <th>국가</th> <th>GDP</th>
  </tr>
  <tr>  ← 데이터 행
    <td>1</td> <td>미국</td> <td>29,184,900</td>
  </tr>
</table>
```

- `tr`: 표의 한 행
- `th`: 제목 셀
- `td`: 데이터 셀
- `find_all(["th", "td"])`: 제목 셀 또는 데이터 셀을 모두 찾기


In [82]:
# 첫 번째 표 선택
table = soup.select_one("table")
print(table)

<table class="wikitable sortable">
<caption>2024년 기준 UN 조사 명목 국내 총생산 순위<sup about="#mwt4" class="mw-ref reference" data-mw='{"name":"ref","attrs":{},"body":{"id":"mw-reference-text-cite_note-2"}}' id="cite_ref-2" rel="dc:references" typeof="mw:Extension/ref"><a href="#cite_note-2" id="mwDw"><span class="mw-reflink-text" id="mwEA"><span class="cite-bracket" id="mwEQ">[</span>2<span class="cite-bracket" id="mwEg">]</span></span></a></sup></caption>
<tbody><tr><th colspan="2">순위</th><th rowspan="2">국가/령</th><th rowspan="2">GDP<sup about="#mwt5" class="mw-ref reference" data-mw='{"name":"ref","attrs":{},"body":{"id":"mw-reference-text-cite_note-3"}}' id="cite_ref-3" rel="dc:references" typeof="mw:Extension/ref"><a href="#cite_note-3" id="mwEw"><span class="mw-reflink-text" id="mwFA"><span class="cite-bracket" id="mwFQ">[</span>3<span class="cite-bracket" id="mwFg">]</span></span></a></sup></th></tr>
<tr>
<th>전체</th><th>국가</th></tr>
<tr>
<td align="center">1</td><td align="center">1</td><td

In [83]:
# 표의 모든 행 선택
rows = table.find_all("tr")
print(rows)

[<tr><th colspan="2">순위</th><th rowspan="2">국가/령</th><th rowspan="2">GDP<sup about="#mwt5" class="mw-ref reference" data-mw='{"name":"ref","attrs":{},"body":{"id":"mw-reference-text-cite_note-3"}}' id="cite_ref-3" rel="dc:references" typeof="mw:Extension/ref"><a href="#cite_note-3" id="mwEw"><span class="mw-reflink-text" id="mwFA"><span class="cite-bracket" id="mwFQ">[</span>3<span class="cite-bracket" id="mwFg">]</span></span></a></sup></th></tr>, <tr>
<th>전체</th><th>국가</th></tr>, <tr>
<td align="center">1</td><td align="center">1</td><td align="left"><span data-sort-value="미국"><span class="flagicon"><span class="mw-image-border" typeof="mw:File"><span><img alt="" class="mw-file-element" data-file-height="650" data-file-type="drawing" data-file-width="1235" decoding="async" height="12" resource="//ko.wikipedia.org/wiki/파일:Flag_of_the_United_States.svg" src="//upload.wikimedia.org/wikipedia/commons/thumb/a/a4/Flag_of_the_United_States.svg/40px-Flag_of_the_United_States.svg.png" srcset=

In [84]:
len(rows)

215

In [86]:
records = []
for tr in rows:
    # th 또는 td 셀의 글자만 추출
    cols = [cell.get_text(strip=True) for cell in tr.find_all(["th", "td"])]
    if cols:
        records.append(cols)

print("추출한 행 수:", len(records))
records[:3]

추출한 행 수: 215


[['순위', '국가/령', 'GDP[3]'],
 ['전체', '국가'],
 ['1', '1', '미국', '29,298,000,000,000']]

In [87]:
# 위키 표 내부 구조에 맞춰 필요한 열만 선택
body = []

for row in records[1:]:
    if len(row) >= 4:
        body.append([row[0], row[2], row[3]])

# DataFrame: 표 형태 데이터로 변환
df_gdp = pd.DataFrame(body, columns=["순위", "국가/령", "GDP"])

df_gdp.head()  # 앞부분 몇 줄 확인


,순위,국가/령,GDP
0,1,미국,"29,298,000,000,000"
1,2,중국,"18,743,802,246,861"
2,3,독일,"4,659,929,336,891"
3,4,일본,"4,026,210,821,147"
4,5,인도,"3,952,244,449,119"
